In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Robust path: works whether you run from project root or notebook/.
here = Path.cwd().resolve()
name = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

candidates = [
    here / "data" / name,
    here.parent / "data" / name,
]

csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        f"Dataset not found. Tried: {', '.join(str(p) for p in candidates)}"
    )

df = pd.read_csv(csv_path)

# Match script cleaning
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

df.head()


### Insight

This dataset includes behavioral and demographic information of telecom customers. It is suitable for analyzing churn patterns and building predictive models to identify at-risk customers.


In [ ]:
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[-0.1, 12, 24, 48, 72, np.inf],
    labels=["0-1y", "1-2y", "2-4y", "4-6y", "6y+"],
    include_lowest=True,
)


### Insight

Converting tenure into grouped categories allows better segmentation of customers by lifecycle stage. This improves interpretability and helps detect churn patterns across early, mid, and long-term customers.


In [ ]:
df.info()


### Insight

The dataset includes both categorical and numerical variables, requiring preprocessing steps such as encoding and type conversion. Minimal missing values suggest a high-quality dataset suitable for supervised learning tasks like churn prediction.


In [ ]:
df.isnull().sum()


### Insight

Missing values are minimal in the dataset, with `TotalCharges` being the only affected feature. These values were treated by type conversion and median imputation to maintain data integrity for machine learning models.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
order = ["0-1y", "1-2y", "2-4y", "4-6y", "6y+"]
sns.countplot(data=df, x="tenure_group", hue="Churn", order=order)
plt.title("Churn by Tenure Group")
plt.tight_layout()
plt.show()


### Insight

Churn is heavily concentrated among new customers (0?1 year tenure), while long-term customers show significantly lower churn rates. This suggests that customer retention improves over time, making early-stage engagement critical.


In [ ]:
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Churn by Contract Type")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


### Insight

Customers with month-to-month contracts exhibit the highest churn rate, while longer-term contracts (one or two years) show better retention. This indicates that contract duration is a strong predictor of customer loyalty.


In [ ]:
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges vs Churn")
plt.tight_layout()
plt.show()


### Insight

There is a clear tendency for churned customers to have higher monthly charges. This indicates that pricing may be a key factor influencing customer retention and churn behavior.


## Machine Learning (Pipeline)

This section trains a logistic regression model using a reproducible scikit-learn `Pipeline` that handles preprocessing (scaling numeric columns + one-hot encoding categorical columns) without manual feature engineering steps.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Prepare data (match src/analysis.py)
data = df.drop(columns=["customerID"]).copy()

data["Churn"] = data["Churn"].map({"No": 0, "Yes": 1})
if data["Churn"].isna().any():
    bad = sorted(set(df["Churn"].dropna().unique()) - {"No", "Yes"})
    raise ValueError(f"Unexpected Churn values: {bad}")

X = data.drop(columns=["Churn"])
y = data["Churn"].astype(int)

cat_cols = X.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
    ],
    remainder="drop",
)

clf = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=2000)),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))
